In [1]:
import pandas as pd
import numpy as np
df_train = pd.read_csv('/content/train.csv')

In [6]:
df_train.columns

Index(['text', 'label'], dtype='object')

In [2]:
df_train['label'].value_counts()

,count
label,
1,5362
0,4666
3,2159
4,1937
2,1304
5,572


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
T = Tokenizer()

In [7]:
T.fit_on_texts(df_train['text'])

In [8]:
T.word_index

{'i': 1,
 'feel': 2,
 'and': 3,
 'to': 4,
 'the': 5,
 'a': 6,
 'feeling': 7,
 'that': 8,
 'of': 9,
 'my': 10,
 'in': 11,
 'it': 12,
 'like': 13,
 'so': 14,
 'for': 15,
 'im': 16,
 'me': 17,
 'but': 18,
 'was': 19,
 'have': 20,
 'is': 21,
 'this': 22,
 'am': 23,
 'with': 24,
 'not': 25,
 'about': 26,
 'be': 27,
 'as': 28,
 'on': 29,
 'you': 30,
 'just': 31,
 'at': 32,
 'when': 33,
 'or': 34,
 'all': 35,
 'because': 36,
 'more': 37,
 'do': 38,
 'can': 39,
 'really': 40,
 'up': 41,
 't': 42,
 'are': 43,
 'by': 44,
 'very': 45,
 'know': 46,
 'been': 47,
 'if': 48,
 'out': 49,
 'myself': 50,
 'time': 51,
 'how': 52,
 'what': 53,
 'get': 54,
 'little': 55,
 'had': 56,
 'now': 57,
 'will': 58,
 'from': 59,
 'being': 60,
 'they': 61,
 'people': 62,
 'them': 63,
 'would': 64,
 'he': 65,
 'want': 66,
 'her': 67,
 'some': 68,
 'think': 69,
 'one': 70,
 'still': 71,
 'ive': 72,
 'him': 73,
 'even': 74,
 'who': 75,
 'an': 76,
 'life': 77,
 'its': 78,
 'make': 79,
 'there': 80,
 'we': 81,
 'bit': 82

In [30]:
vocab_size = len(T.word_index)
print("Size of the vocabulary: ",vocab_size)

Size of the vocabulary:  15212


In [9]:
sentences_list = df_train['text'].tolist()
print(len(sentences_list))

16000


In [16]:
max_length = max(len(x.split()) for x in sentences_list)
print("Maximum length of sentence:",max_length)

longest_sentence = max(sentences_list, key=len)
print("Longest sentence:", longest_sentence)
print("Index:", sentences_list.index(longest_sentence))

Maximum length of sentence: 66
Longest sentence: i hope that those of you who actauly found this and read it feel possibly inspired to go out and buy some of these items or even go through storage and see what clothes of yours your mom saved and that you still have a hope of fitting in and mix up your wardrobe for this summer and have a little fun
Index: 10390


In [18]:
sequences = T.texts_to_sequences(sentences_list)

In [20]:
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='pre')
padded_sequences

array([[   0,    0,    0, ...,  138,    2,  678],
       [   0,    0,    0, ...,    3,   21, 1254],
       [   0,    0,    0, ...,    2,  494,  437],
       ...,
       [   0,    0,    0, ...,    3,  101, 1331],
       [   0,    0,    0, ...,  339,    8,   42],
       [   0,    0,    0, ...,   25, 3585,   12]], dtype=int32)

In [21]:
print(type(padded_sequences))

<class 'numpy.ndarray'>


In [24]:
X = padded_sequences
X.shape

(16000, 66)

In [25]:
y = df_train['label']
y.shape

(16000,)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
model = Sequential()
model.add(Embedding(input_dim=15213, output_dim=8, input_length=max_length))
model.add(LSTM(64))
model.add(Dense(6, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [35]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [36]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
model.fit(X,y,epochs=50)

Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 18s 31ms/step - accuracy: 0.3988 - loss: 1.5084
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - accuracy: 0.7126 - loss: 0.8127
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - accuracy: 0.8257 - loss: 0.4816
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.8864 - loss: 0.3153
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - accuracy: 0.9352 - loss: 0.1967
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - accuracy: 0.9587 - loss: 0.1371
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.9674 - loss: 0.1052
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.9758 - loss: 0.0800
Epoch 9/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - accuracy: 0.9818 - loss: 0.0616
Epoch 10/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 31ms/step - accuracy: 0.9831 - loss: 0.0559
Epoch 11/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - accuracy: 0.9818 - loss: 0.0565
Epoch 12/50
500/500 ━━━━━━━━━━

In [38]:
model.evaluate(X_test, y_test)

100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9987 - loss: 0.0043


[0.004288120660930872, 0.9987499713897705]

In [40]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

def predict_emotion(sentence, tokenizer, model, max_length, label_map):
    seq = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=max_length, padding='pre')
    pred = model.predict(padded, verbose=0)
    pred_class = np.argmax(pred, axis=1)[0]
    return {
        "sentence": sentence,
        "processed_sequence": padded,
        "prediction_index": pred_class,
        "prediction_label": label_map[pred_class],
        "prediction_probabilities": pred[0]
    }

In [61]:
text = "i find myself in the odd position of feeling supportive of"
result = predict_emotion(text, T, model, max_length, label_map)

print("Sentence:", result["sentence"])
print("Predicted class index:", result["prediction_index"])
print("Predicted emotion:", result["prediction_label"])

Sentence: i find myself in the odd position of feeling supportive of
Predicted class index: 2
Predicted emotion: love


#### Testing on test.csv

In [62]:
df_test = pd.read_csv('/content/test.csv')
df_test.head()

,text,label
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


#### Preprocess test text with the same tokenizer

In [63]:
test_sentences = df_test['text'].tolist()
test_sequences = T.texts_to_sequences(test_sentences)
X_new_test = pad_sequences(test_sequences, maxlen=max_length, padding='pre')

#### Labels

In [64]:
y_new_test = df_test['label'].values

#### Evaluate

In [70]:
X_new_test = np.asarray(X_new_test, dtype=np.int32)
y_new_test = np.asarray(y_new_test, dtype=np.int32)

loss, accuracy = model.evaluate(X_new_test, y_new_test, verbose=1)
print("Loss:", loss)
print("Accuracy:", accuracy)

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9080 - loss: 0.4397
Loss: 0.43965256214141846
Accuracy: 0.9079999923706055


In [71]:
model.save("emotion_detection_model.keras")

In [72]:
from google.colab import drive
import pickle
drive.mount('/content/drive')

model.save("/content/drive/MyDrive/emotion_model.keras")

with open("/content/drive/MyDrive/tokenizer.pkl", "wb") as f:
    pickle.dump(T, f)

Mounted at /content/drive
